# Câu B


## TIỀN XỬ LÝ DỮ LIỆU

In [18]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import underthesea   # pip install underthesea
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [19]:
Path = os.path.join(os.getcwd(), "Data")

def read_file(filename):
    filename = os.path.join(Path, filename)
    with open(filename, 'r', encoding="utf-16") as f:
        data = f.read()
    return data

def natural_key(name):
    m = re.match(r"(.+?)_ \((\d+)\)\.txt$", name)
    return (m.group(1), int(m.group(2))) if m else (name, 0)

file_names = sorted([f for f in os.listdir(Path) if f.endswith(".txt")], key=natural_key)

Raw_Data = {}      
errors = []
for name in file_names:
    try:
        Raw_Data[name] = read_file(name)
    except Exception as e:
        errors.append((name, str(e)))

print(f"Đã đọc: {len(Raw_Data)} file | Lỗi: {len(errors)}")
for name, err in errors:
    print("  ✗", name, "-", err)

Đã đọc: 475 file | Lỗi: 0


In [20]:
StopWords_File = os.path.join(os.getcwd(), "vietnamese-stopwords.txt")

if os.path.exists(StopWords_File):
    with open(StopWords_File, "r", encoding="utf-8") as f:
        List_StopWords = set(w.strip().lower().replace(" ", "_")
                             for w in f.read().split("\n") if w.strip())
    
else:
    List_StopWords = set()
   

In [21]:
def Text_Preprocessing(doc):
    text_pre = doc.lower()                                   # Chuyển chữ thường

    text_pre = underthesea.word_tokenize(text_pre, format="text")   # Tách từ tiếng Việt
    text_pre = underthesea.text_normalize(text_pre)                 # Chuẩn hóa dấu tiếng Việt

    text_pre = text_pre.replace(u'�', '')               # Xóa ký tự lỗi
    text_pre = re.sub(r'http\S+', '', text_pre)              # Xóa URL có http
    text_pre = re.sub(r'www\S+', '', text_pre)               # Xóa URL có www
    text_pre = re.sub(r"\S*@\S*\s?", " ", text_pre)          # Xóa email
    text_pre = re.sub(r'[^\w\s]', ' ', text_pre)             # Xóa dấu câu, ký tự đặc biệt (giữ dấu _)
    text_pre = re.sub(r"\S*\d\S*", " ", text_pre)            # Xóa số và từ dính số
    text_pre = re.sub(r'\s+', ' ', text_pre).strip()         # Xóa khoảng trắng thừa, xuống dòng

    # Loại bỏ stopwords
    if List_StopWords:
        text_pre = " ".join(w for w in text_pre.split() if w not in List_StopWords)

    return text_pre

In [22]:
Data = {}          
for name, content in Raw_Data.items():
    Data[name] = Text_Preprocessing(content)


empty_docs = [n for n, t in Data.items() if not t.strip()]
for n in empty_docs:
    del Data[n]
print(f"Số văn bản sau tiền xử lý: {len(Data)}" )


Số văn bản sau tiền xử lý: 475


In [23]:
file_list = list(Data.keys())        
corpus    = list(Data.values())      

tfidf_vectorizer = TfidfVectorizer(max_df=0.5, min_df=2)
X = tfidf_vectorizer.fit_transform(corpus)
terms = tfidf_vectorizer.get_feature_names_out()
print("Kích thước ma trận TF-IDF:", X.shape)

Kích thước ma trận TF-IDF: (475, 4836)


## K-MEANS


In [24]:
num_clusters = 6     

km = KMeans(n_clusters=num_clusters, init='k-means++', max_iter=300, n_init=10, random_state=42)
km.fit(X)
clusters = km.labels_.tolist()

In [25]:
print(f"Số cụm: {num_clusters}")
print("Phân cụm:")
for c in range(num_clusters):
    files_in_c = [file_list[i] for i, lb in enumerate(clusters) if lb == c]
    print(f"Cụm {c+1}: " + ", ".join(files_in_c))
    print()

Số cụm: 6
Phân cụm:
Cụm 1: BDS_TN_ (220).txt, BDS_TN_ (221).txt, BDS_TN_ (223).txt, BDS_TN_ (226).txt, BDS_TN_ (227).txt, BDS_TN_ (231).txt, BDS_TN_ (232).txt, BDS_TN_ (233).txt, BDS_TN_ (236).txt, BDS_TN_ (237).txt, BDS_TN_ (243).txt, BDS_TN_ (246).txt, BDS_VNE_ (4).txt, BDS_VNE_ (5).txt, BDS_VNE_ (7).txt, BDS_VNE_ (8).txt, BDS_VNE_ (10).txt, BDS_VNE_ (11).txt, BDS_VNE_ (13).txt, BDS_VNE_ (17).txt, BDS_VNE_ (18).txt, BDS_VNE_ (19).txt, BDS_VNE_ (21).txt, BDS_VNE_ (22).txt, BDS_VNE_ (26).txt, BDS_VNE_ (27).txt, BDS_VNE_ (29).txt, BDS_VNE_ (33).txt, BDS_VNE_ (34).txt, BDS_VNE_ (40).txt, BDS_VNE_ (41).txt, BDS_VNE_ (43).txt, BDS_VNE_ (46).txt, BDS_VNE_ (47).txt, BDS_VNE_ (48).txt, BDS_VNE_ (50).txt, BDS_VNE_ (51).txt, BDS_VNE_ (52).txt, BDS_VNE_ (53).txt, BDS_VNE_ (56).txt, BDS_VNE_ (57).txt, BDS_VNE_ (59).txt, BDS_VNE_ (60).txt, BDS_VNE_ (62).txt, BDS_VNE_ (64).txt, BDS_VNE_ (67).txt

Cụm 2: GDu_NLD_ (589).txt, GDu_NLD_ (590).txt, GDu_NLD_ (591).txt, GDu_NLD_ (592).txt, GDu_NLD_ (593).t

In [26]:
frame = pd.DataFrame({'file': file_list, 'cluster': [c+1 for c in clusters]})
print("Numbers of Docs per cluster:")
print(frame['cluster'].value_counts().sort_index().rename_axis('Cụm').to_string())

Numbers of Docs per cluster:
Cụm
1     46
2     93
3    115
4     90
5     76
6     55
